In [2]:
! uv pip install langchain openai tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Using Python 3.12.10 environment at: c:\Users\kasar\LLMOPS_series\.venv
Resolved 72 packages in 822ms
 Downloaded openai
Prepared 2 packages in 2.72s
Installed 17 packages in 1.55s
 + distro==1.9.0
 + flatbuffers==25.12.19
 + jiter==0.13.0
 + mpmath==1.3.0
 + onnxruntime==1.24.3
 + openai==2.26.0
 + opencv-python==4.13.0.92
 + pillow==12.1.1
 + protobuf==7.34.0
 + pyclipper==1.4.0
 + rapidocr-onnxruntime==1.4.4
 + regex==2026.2.28
 + shapely==2.1.2
 + sniffio==1.3.1
 + sympy==1.14.0
 + tiktoken==0.12.0
 + tqdm==4.67.3


In [2]:
import os

from dotenv import load_dotenv

load_dotenv()


os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


## Data Ingestion


In [3]:
from langchain.document_loaders import TextLoader

In [4]:
loader = TextLoader("C://Users//kasar//LLMOPS_series//data//Agentic AI.txt", encoding="utf8")
documents = loader.load()

In [5]:
documents[0].page_content[:500]  # Print the first 500 characters of the first documen

'Agentic AI is a type of AI system that can act autonomously to achieve goals, not just respond to prompts. It can plan, decide, use tools, and execute multiple steps to complete a task with minimal human input.\n\nIn simple terms:\n\nTraditional AI: responds to a question\nAgentic AI: figures out how to solve a problem and performs actions to achieve the goal.\n\nKey Idea\n\nAgentic AI systems behave like digital agents that can:\n\nUnderstand a goal\n\nPlan steps to achieve it\n\nUse tools or APIs\n\nExecute ta'

In [6]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [10]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=20)

In [12]:
text_chunks=text_splitter.split_documents(documents)

In [13]:
text_chunks

[Document(metadata={'source': 'C://Users//kasar//LLMOPS_series//data//Agentic AI.txt'}, page_content='Agentic AI is a type of AI system that can act autonomously to achieve goals, not just respond to prompts. It can plan, decide, use tools, and execute multiple steps to complete a task with minimal'),
 Document(metadata={'source': 'C://Users//kasar//LLMOPS_series//data//Agentic AI.txt'}, page_content='a task with minimal human input.'),
 Document(metadata={'source': 'C://Users//kasar//LLMOPS_series//data//Agentic AI.txt'}, page_content='In simple terms:\n\nTraditional AI: responds to a question\nAgentic AI: figures out how to solve a problem and performs actions to achieve the goal.\n\nKey Idea'),
 Document(metadata={'source': 'C://Users//kasar//LLMOPS_series//data//Agentic AI.txt'}, page_content='Key Idea\n\nAgentic AI systems behave like digital agents that can:\n\nUnderstand a goal\n\nPlan steps to achieve it\n\nUse tools or APIs\n\nExecute tasks\n\nEvaluate results and adjust'),
 D

In [14]:
! uv pip install faiss-cpu


Using Python 3.13.12 environment at: C:\Users\kasar\LLMOPS_series\.venv
Audited 1 package in 277ms


In [15]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [16]:
embeddings=OpenAIEmbeddings()

C:\Users\kasar\AppData\Local\Temp\ipykernel_30576\2276573386.py:1: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings=OpenAIEmbeddings()


In [17]:
vectorstore=FAISS.from_documents(text_chunks, embeddings)

In [18]:
vectorstore

In [19]:
retriever=vectorstore.as_retriever()

In [20]:
# Perform similarity search
query = "What is the Key Characteristics of Agentic AI?"
docs = vectorstore.similarity_search(query, k=4)

# Display the results
for i, doc in enumerate(docs):
    print(f"Document {i+1}:")
    print(doc.page_content)
    print("-" * 50)


Document 1:
Key Idea

Agentic AI systems behave like digital agents that can:

Understand a goal

Plan steps to achieve it

Use tools or APIs

Execute tasks

Evaluate results and adjust
--------------------------------------------------
Document 2:
In simple terms:

Traditional AI: responds to a question
Agentic AI: figures out how to solve a problem and performs actions to achieve the goal.

Key Idea
--------------------------------------------------
Document 3:
Agentic AI is a type of AI system that can act autonomously to achieve goals, not just respond to prompts. It can plan, decide, use tools, and execute multiple steps to complete a task with minimal
--------------------------------------------------
Document 4:
Agentic AI

User:

"Find the best laptop under $1500 and email me the top 3 deals."

Agentic AI will:

Search the internet

Compare prices

Analyze reviews

Pick top options
--------------------------------------------------


In [21]:
from langchain.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [22]:
prompt=ChatPromptTemplate.from_template(template)

In [23]:
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse ten sentences maximum and keep the answer concise.\nQuestion: {question}\nContext: {context}\nAnswer:\n"), additional_kwargs={})])

In [24]:
from langchain.schema.output_parser import StrOutputParser

In [25]:
output_parser=StrOutputParser()

In [26]:
from langchain.chat_models import ChatOpenAI

llm_model=ChatOpenAI(model_name="gpt-4o-mini")

C:\Users\kasar\AppData\Local\Temp\ipykernel_30576\2671044564.py:3: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm_model=ChatOpenAI(model_name="gpt-4o-mini")


In [27]:
from langchain.schema.runnable import RunnablePassthrough


rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()}
    | prompt
    | llm_model
    | output_parser
)

In [28]:
rag_chain.invoke("tell me about Agentic AI")

'Agentic AI refers to a type of artificial intelligence that can operate autonomously to achieve specific goals, rather than merely responding to prompts. These systems function as digital agents capable of understanding a goal, planning the steps needed to achieve it, using tools or APIs, executing tasks, and evaluating results to make adjustments. For example, if asked to find the best laptop under $1500, an Agentic AI would search the internet, compare prices, analyze reviews, and provide the top options. This contrasts with traditional AI, which typically only responds to questions without taking further action. Overall, Agentic AI represents a more advanced and proactive approach to problem-solving in the realm of artificial intelligence.'